# Spam SMS Text Classifier Using NLP and Naive Bayes
### End-to-End Data Science & Machine Learning Pipeline
**Dataset:** UCI SMS Spam Collection  
**Techniques:** NLP Preprocessing, TF-IDF Vectorization, Numerical Feature Engineering, Multinomial & Complement Naive Bayes, Precision-Driven Evaluation


## 1. Environment Setup & Library Imports


In [ ]:
import os
import sys
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# Set visualization style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Download NLTK corpora
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Libraries imported successfully.')


## 2. Dataset Loading & Initial Inspection
We load the **UCI SMS Spam Collection** dataset. The raw dataset contains tab-separated entries labelled as `ham` or `spam`.


In [ ]:
data_path = '../data/SMSSpamCollection'
if not os.path.exists(data_path):
    data_path = 'data/SMSSpamCollection'

df_raw = pd.read_csv(data_path, sep='\t', names=['label', 'message'], quoting=3, encoding='utf-8')
print('Raw dataset shape:', df_raw.shape)
print('\nLabel Distribution (Raw):')
print(df_raw['label'].value_counts())
print('\nMissing values in dataset:', df_raw.isnull().sum().to_dict())
print('Duplicate messages:', df_raw.duplicated(subset=['message']).sum())

# Display sample records
df_raw.head(10)


## 3. Data Cleaning & Deduplication
Duplicate SMS messages can artificially skew training and test distributions. We remove duplicate messages to ensure rigorous evaluation.


In [ ]:
df = df_raw.drop_duplicates(subset=['message']).reset_index(drop=True)
print('Cleaned dataset shape (duplicates removed):', df.shape)
print('\nLabel counts after deduplication:')
print(df['label'].value_counts())
spam_pct = (df['label'] == 'spam').mean() * 100
ham_pct = (df['label'] == 'ham').mean() * 100
print(f'HAM: {ham_pct:.2f}% | SPAM: {spam_pct:.2f}%')


## 4. Hand-Crafted Numerical Feature Extraction
In spam detection, message structure (capital letters, currency symbols, punctuation bursts, URLs, and lengths) provides strong non-textual signals. We extract 14 distinct numerical features.


In [ ]:
CURRENCY_REGEX = re.compile(r'[\$£€₹]')
URL_REGEX = re.compile(r'https?://\S+|www\.\S+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}|bit\.ly\S*|goo\.gl\S*')
REPEATED_CHAR_REGEX = re.compile(r'(.)\1{2,}')

def extract_features(text):
    if not isinstance(text, str):
        text = '' if text is None else str(text)
    msg_len = len(text)
    words = text.split()
    word_cnt = len(words)
    sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    sentence_cnt = len(sentences) if len(sentences) > 0 else (1 if msg_len > 0 else 0)
    punct_cnt = sum(1 for c in text if c in string.punctuation)
    punct_ratio = (punct_cnt / msg_len) if msg_len > 0 else 0.0
    alpha_cnt = sum(1 for c in text if c.isalpha())
    upper_cnt = sum(1 for c in text if c.isupper())
    cap_ratio = (upper_cnt / alpha_cnt) if alpha_cnt > 0 else 0.0
    digit_cnt = sum(1 for c in text if c.isdigit())
    digit_ratio = (digit_cnt / msg_len) if msg_len > 0 else 0.0
    has_url = 1.0 if URL_REGEX.search(text) else 0.0
    currency_cnt = len(CURRENCY_REGEX.findall(text))
    exclamation_cnt = text.count('!')
    question_cnt = text.count('?')
    repeated_chars = len(REPEATED_CHAR_REGEX.findall(text))
    
    return {
        'message_length': float(msg_len),
        'word_count': float(word_cnt),
        'sentence_count': float(sentence_cnt),
        'punctuation_count': float(punct_cnt),
        'punctuation_ratio': float(punct_ratio),
        'uppercase_count': float(upper_cnt),
        'capitalization_ratio': float(cap_ratio),
        'digit_count': float(digit_cnt),
        'digit_ratio': float(digit_ratio),
        'contains_url': float(has_url),
        'currency_count': float(currency_cnt),
        'exclamation_count': float(exclamation_cnt),
        'question_count': float(question_cnt),
        'repeated_char_count': float(repeated_chars)
    }

num_features_df = pd.DataFrame([extract_features(m) for m in df['message']])
df = pd.concat([df, num_features_df], axis=1)
df['target'] = df['label'].map({'ham': 0, 'spam': 1})
df.head(5)


## 5. Exploratory Data Analysis (EDA) Visualizations
We generate statistical visualizations to understand how spam and ham messages differ across textual and numerical dimensions.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Class Distribution
sns.countplot(data=df, x='label', ax=axes[0, 0], palette=['#2b8a3e', '#e03131'])
axes[0, 0].set_title('1. Class Distribution (Ham vs Spam)')
axes[0, 0].set_xlabel('Class')
axes[0, 0].set_ylabel('Number of Messages')

# 2. Message Length Distribution
sns.histplot(data=df, x='message_length', hue='label', bins=50, kde=True, ax=axes[0, 1], palette=['#2b8a3e', '#e03131'])
axes[0, 1].set_title('2. Message Length Distribution')
axes[0, 1].set_xlim(0, 300)

# 3. Average Word Count by Class
sns.barplot(data=df, x='label', y='word_count', ax=axes[0, 2], palette=['#2b8a3e', '#e03131'])
axes[0, 2].set_title('3. Average Word Count by Class')

# 4. Punctuation Ratio Comparison
sns.boxplot(data=df, x='label', y='punctuation_ratio', ax=axes[1, 0], palette=['#2b8a3e', '#e03131'])
axes[1, 0].set_title('4. Punctuation Ratio Comparison')

# 5. Capitalization Ratio Comparison
sns.boxplot(data=df, x='label', y='capitalization_ratio', ax=axes[1, 1], palette=['#2b8a3e', '#e03131'])
axes[1, 1].set_title('5. Capitalization Ratio Comparison')

# 6. Digit Count Comparison
sns.barplot(data=df, x='label', y='digit_count', ax=axes[1, 2], palette=['#2b8a3e', '#e03131'])
axes[1, 2].set_title('6. Average Digit Count by Class')

plt.tight_layout()
plt.show()


## 6. NLP Preprocessing Pipeline
Text preprocessing involves:
1. Converting text to lowercase
2. Removing URLs, special characters, and extra whitespaces
3. Tokenizing
4. Removing English stopwords
5. Applying Porter Stemming


In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[\$£€₹]', ' currency_symbol ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    stemmed = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(stemmed)

df['clean_message'] = df['message'].apply(clean_text)
print('Sample clean messages:')
for i in range(3):
    print('Raw:  ', df['message'].iloc[i])
    print('Clean:', df['clean_message'].iloc[i])
    print('-' * 60)


## 7. Train-Test Split with Stratification
We split the dataset into 80% training and 20% testing sets using stratification on the target label to preserve the spam/ham class ratio.


In [ ]:
feature_cols = [
    'message_length', 'word_count', 'sentence_count', 'punctuation_count',
    'punctuation_ratio', 'uppercase_count', 'capitalization_ratio', 'digit_count',
    'digit_ratio', 'contains_url', 'currency_count', 'exclamation_count',
    'question_count', 'repeated_char_count'
]

X_train_text, X_test_text, y_train, y_test, num_train, num_test = train_test_split(
    df['clean_message'],
    df['target'],
    df[feature_cols],
    test_size=0.20,
    random_state=42,
    stratify=df['target']
)

print(f'Training samples: {len(y_train)} (Spam: {y_train.sum()}, Ham: {len(y_train) - y_train.sum()})')
print(f'Testing samples:  {len(y_test)} (Spam: {y_test.sum()}, Ham: {len(y_test) - y_test.sum()})')


## 8. TF-IDF Vectorization & Feature Scaling
We fit a TF-IDF vectorizer (unigrams + bigrams, min_df=2, max_df=0.95, sublinear_tf=True) **strictly on training data** to prevent data leakage. Numerical features are scaled to [0, 1] using MinMaxScaler and combined with the sparse TF-IDF matrix.


In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

scaler = MinMaxScaler()
num_train_scaled = scaler.fit_transform(num_train)
num_test_scaled = scaler.transform(num_test)

X_train_comb = hstack([X_train_tfidf, num_train_scaled])
X_test_comb = hstack([X_test_tfidf, num_test_scaled])

print(f'TF-IDF vocabulary size: {X_train_tfidf.shape[1]}')
print(f'Combined feature space shape: {X_train_comb.shape}')


## 9. Naive Bayes Model Comparison & Hyperparameter Tuning
We benchmark **MultinomialNB** vs **ComplementNB** across smoothing parameters (alpha).


In [ ]:
alphas = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
results = []

for alpha in alphas:
    for model_cls, model_name in [(MultinomialNB, 'MultinomialNB'), (ComplementNB, 'ComplementNB')]:
        clf = model_cls(alpha=alpha)
        clf.fit(X_train_comb, y_train)
        y_pred = clf.predict(X_test_comb)
        
        results.append({
            'Model': model_name,
            'Alpha': alpha,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Spam_Precision': precision_score(y_test, y_pred, pos_label=1),
            'Spam_Recall': recall_score(y_test, y_pred, pos_label=1),
            'Spam_F1': f1_score(y_test, y_pred, pos_label=1)
        })

df_results = pd.DataFrame(results)
print(df_results.sort_values(by=['Spam_Precision', 'Spam_F1'], ascending=False).to_string(index=False))


## 10. Final Model Evaluation & Precision-Driven Metrics
We train our final selected model (`MultinomialNB(alpha=0.2)`) and evaluate it thoroughly on the unseen test set.


In [ ]:
best_model = MultinomialNB(alpha=0.2)
best_model.fit(X_train_comb, y_train)

y_pred = best_model.predict(X_test_comb)
y_prob = best_model.predict_proba(X_test_comb)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec_spam = precision_score(y_test, y_pred, pos_label=1)
rec_spam = recall_score(y_test, y_pred, pos_label=1)
f1_s = f1_score(y_test, y_pred, pos_label=1)

print('=' * 45)
print('FINAL MODEL PERFORMANCE ON TEST SET:')
print(f'Accuracy:       {acc * 100:.2f}%')
print(f'Spam Precision: {prec_spam * 100:.2f}%')
print(f'Spam Recall:    {rec_spam * 100:.2f}%')
print(f'Spam F1-Score:  {f1_s * 100:.2f}%')
print('=' * 45)
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['HAM', 'SPAM']))


## 11. Confusion Matrix Heatmap & ROC Curve


In [ ]:
cm = confusion_matrix(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)
pr_prec, pr_rec, _ = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Predicted HAM', 'Predicted SPAM'],
            yticklabels=['Actual HAM', 'Actual SPAM'])
axes[0].set_title(f'Confusion Matrix\n(TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]})')

# ROC Curve
axes[1].plot(fpr, tpr, color='#1c7ed6', lw=2, label=f'ROC Curve (AUC = {roc_auc_val:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

# Precision-Recall Curve
axes[2].plot(pr_rec, pr_prec, color='#e03131', lw=2, label='Precision-Recall Curve')
axes[2].set_title('Precision-Recall Curve')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(loc='lower left')

plt.tight_layout()
plt.show()


## 12. Error Analysis: Misclassified Messages
Examining False Positives and False Negatives reveals edge cases where spam indicators overlap with legitimate casual conversations.


In [ ]:
test_df = df.iloc[y_test.index].copy()
test_df['actual'] = y_test
test_df['predicted'] = y_pred
test_df['spam_prob'] = y_prob

print('=== FALSE POSITIVES (Actual HAM predicted as SPAM) ===')
fp_df = test_df[(test_df['actual'] == 0) & (test_df['predicted'] == 1)]
for _, row in fp_df.iterrows():
    print(f'P(Spam)={row["spam_prob"]*100:.1f}% | Message: {row["message"]}')

print('\n=== SAMPLE FALSE NEGATIVES (Actual SPAM predicted as HAM) ===')
fn_df = test_df[(test_df['actual'] == 1) & (test_df['predicted'] == 0)].head(5)
for _, row in fn_df.iterrows():
    print(f'P(Spam)={row["spam_prob"]*100:.1f}% | Message: {row["message"]}')


## 13. Interactive SMS Classifier Function


In [ ]:
def test_sms(msg):
    num_f = extract_features(msg)
    num_scaled = scaler.transform(pd.DataFrame([num_f])[feature_cols])
    clean = clean_text(msg)
    vec = tfidf.transform([clean])
    comb = hstack([vec, num_scaled])
    pred = best_model.predict(comb)[0]
    prob = best_model.predict_proba(comb)[0]
    label = 'SPAM' if pred == 1 else 'HAM'
    print(f'Message: "{msg}"')
    print(f'Result:  {label} (HAM: {prob[0]*100:.2f}%, SPAM: {prob[1]*100:.2f}%)')
    print('-' * 60)

test_sms('Congratulations! You have won a free prize. Call now to claim!')
test_sms('Hey bro, are we still meeting for lunch at 1pm?')
test_sms('URGENT: Your bank account has been compromised. Verify at http://bit.ly/bank-security immediately!')
